In [ ]:
import sys
from pathlib import Path

# Get the root directory of the project
project_root = Path("../../KCFM_tutorial/cell_atlases").parent.parent
# project_root = Path(__file__).parent.parent
sys.path.append(str(project_root))
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [ ]:
import os
import pickle
import torch
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
from models.gene_tokenizer import GeneVocab
from models.model import MambaModel

In [ ]:
# configuration parameters
max_seq_len = 2049
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Cell type mapping
celltype_to_id = {
    'B cell': 0,
    'Bergmann glial cell': 1,
    'CD14-low, CD16-positive monocyte': 2,
    'CD14-positive monocyte': 3,
    'CD14-positive, CD16-negative classical monocyte': 4,
    'CD14-positive, CD16-positive monocyte': 5,
    'CD16-negative, CD56-bright natural killer cell, human': 6,
    'CD16-positive, CD56-dim natural killer cell, human': 7,
    'CD1c-positive myeloid dendritic cell': 8,
    'CD4-positive helper T cell': 9,
    'CD4-positive, alpha-beta T cell': 10,
    'CD4-positive, alpha-beta cytotoxic T cell': 11,
    'CD4-positive, alpha-beta memory T cell': 12,
    'CD8-alpha-alpha-positive, alpha-beta intraepithelial T cell': 13,
    'CD8-positive, alpha-beta T cell': 14,
    'CD8-positive, alpha-beta cytotoxic T cell': 15,
    'CD8-positive, alpha-beta memory T cell': 16,
    'IgA plasma cell': 17,
    'IgG plasma cell': 18,
    'L2/3-6 intratelencephalic projecting glutamatergic cortical neuron': 19,
    'L6b glutamatergic cortical neuron': 20,
    'Schwann cell': 21,
    'T cell': 22,
    'T follicular helper cell': 23,
    'T-helper 17 cell': 24,
    'T-helper 22 cell': 25,
    'activated CD4-positive, alpha-beta T cell': 26,
    'activated CD8-positive, alpha-beta T cell': 27,
    'alpha-beta T cell': 28,
    'alternatively activated macrophage': 29,
    'alveolar macrophage': 30,
    'alveolar type 1 fibroblast cell': 31,
    'alveolar type 2 fibroblast cell': 32,
    'amacrine cell': 33,
    'astrocyte': 34,
    'astrocyte of the cerebral cortex': 35,
    'basal cell': 36,
    'basal cell of epithelium of trachea': 37,
    'blood vessel endothelial cell': 38,
    'bronchus fibroblast of lung': 39,
    'capillary endothelial cell': 40,
    'cardiac muscle cell': 41,
    'cardiac neuron': 42,
    'caudal ganglionic eminence derived GABAergic cortical interneuron': 43,
    'central memory CD4-positive, alpha-beta T cell': 44,
    'central memory CD8-positive, alpha-beta T cell': 45,
    'central nervous system macrophage': 46,
    'chandelier pvalb GABAergic cortical interneuron': 47,
    'ciliated columnar cell of tracheobronchial tree': 48,
    'class switched memory B cell': 49,
    'classical monocyte': 50,
    'club cell': 51,
    'conventional dendritic cell': 52,
    'corticothalamic-projecting glutamatergic cortical neuron': 53,
    'dendritic cell': 54,
    'double negative T regulatory cell': 55,
    'double negative thymocyte': 56,
    'double-positive, alpha-beta thymocyte': 57,
    'effector CD8-positive, alpha-beta T cell': 58,
    'effector memory CD4-positive, alpha-beta T cell': 59,
    'effector memory CD8-positive, alpha-beta T cell': 60,
    'effector memory CD8-positive, alpha-beta T cell, terminally differentiated': 61,
    'elicited macrophage': 62,
    'endothelial cell': 63,
    'endothelial cell of artery': 64,
    'endothelial cell of lymphatic vessel': 65,
    'enteric smooth muscle cell': 66,
    'enterocyte': 67,
    'enteroendocrine cell': 68,
    'ependymal cell': 69,
    'epithelial cell of proximal tubule': 70,
    'erythroblast': 71,
    'erythrocyte': 72,
    'exhausted T cell': 73,
    'fallopian tube secretory epithelial cell': 74,
    'fibroblast of cardiac tissue': 75,
    'foveolar cell of stomach': 76,
    'gamma-delta T cell': 77,
    'glutamatergic neuron': 78,
    'goblet cell': 79,
    'granulocyte': 80,
    'granulosa cell': 81,
    'hematopoietic stem cell': 82,
    'immature B cell': 83,
    'immature innate lymphoid cell': 84,
    'inflammatory macrophage': 85,
    'innate lymphoid cell': 86,
    'intermediate monocyte': 87,
    'intestine goblet cell': 88,
    'intraepithelial lymphocyte': 89,
    'keratinocyte': 90,
    'kidney collecting duct intercalated cell': 91,
    'kidney collecting duct principal cell': 92,
    'kidney connecting tubule epithelial cell': 93,
    'kidney distal convoluted tubule epithelial cell': 94,
    'kidney interstitial fibroblast': 95,
    'kidney loop of Henle thick ascending limb epithelial cell': 96,
    'kidney loop of Henle thin ascending limb epithelial cell': 97,
    'kidney loop of Henle thin descending limb epithelial cell': 98,
    'lamp5 GABAergic cortical interneuron': 99,
    'leukocyte': 100,
    'luminal epithelial cell of mammary gland': 101,
    'lung macrophage': 102,
    'lung pericyte': 103,
    'lymphocyte': 104,
    'lymphoid lineage restricted progenitor cell': 105,
    'macrophage': 106,
    'mast cell': 107,
    'mature B cell': 108,
    'mature NK T cell': 109,
    'mature T cell': 110,
    'mature alpha-beta T cell': 111,
    'mature gamma-delta T cell': 112,
    'megakaryocyte-erythroid progenitor cell': 113,
    'memory B cell': 114,
    'memory T cell': 115,
    'mesothelial cell': 116,
    'microglial cell': 117,
    'monocyte': 118,
    'mucosal invariant T cell': 119,
    'mucous neck cell': 120,
    'myoepithelial cell of mammary gland': 121,
    'naive B cell': 122,
    'naive T cell': 123,
    'naive thymus-derived CD4-positive, alpha-beta T cell': 124,
    'naive thymus-derived CD8-positive, alpha-beta T cell': 125,
    'nasal mucosa goblet cell': 126,
    'natural killer cell': 127,
    'near-projecting glutamatergic cortical neuron': 128,
    'neuron': 129,
    'neutrophil': 130,
    'non-classical monocyte': 131,
    'oligodendrocyte': 132,
    'oligodendrocyte precursor cell': 133,
    'paneth cell': 134,
    'pericyte': 135,
    'plasma cell': 136,
    'plasmablast': 137,
    'plasmacytoid dendritic cell': 138,
    'platelet': 139,
    'precursor B cell': 140,
    'pro-B cell': 141,
    'promonocyte': 142,
    'pulmonary artery endothelial cell': 143,
    'pvalb GABAergic cortical interneuron': 144,
    'regulatory T cell': 145,
    'renal interstitial pericyte': 146,
    'respiratory basal cell': 147,
    'respiratory hillock cell': 148,
    'retina horizontal cell': 149,
    'retinal cone cell': 150,
    'retinal ganglion cell': 151,
    'retinal rod cell': 152,
    'smooth muscle cell': 153,
    'sncg GABAergic cortical interneuron': 154,
    'sst GABAergic cortical interneuron': 155,
    'tracheal goblet cell': 156,
    'tracheobronchial smooth muscle cell': 157,
    'transitional stage B cell': 158,
    'type I pneumocyte': 159,
    'type II pneumocyte': 160,
    'vascular associated smooth muscle cell': 161,
    'vein endothelial cell': 162,
    'vip GABAergic cortical interneuron': 163
}
id_to_celltype = {v: k for k, v in celltype_to_id.items()}
class_num = len(celltype_to_id)

In [ ]:
class SingleParquetFileDataset(Dataset):
    """Read the dataset of a single parquet file"""
    def __init__(self, file_path):
        self.df = pq.read_table(file_path).to_pandas()
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            'X': torch.FloatTensor(row['X']),
            'cell_type': row['cell_type']
        }

In [ ]:
def preprocess_batch(x_batch, feature_names, vocab, device):
    """Preprocess batch data into model input format"""
    batch_size = x_batch.size(0)
    values_tensor = torch.full((batch_size, max_seq_len), -2.0, device=device)
    src_tensor = torch.full((batch_size, max_seq_len), vocab["<pad>"], dtype=torch.long, device=device)
    padding_mask = torch.zeros((batch_size, max_seq_len), dtype=torch.bool, device=device)
    
    nonzero_mask = x_batch > 0
    nonzero_counts = nonzero_mask.sum(dim=1)
    
    for i in range(batch_size):
        nz_count = min(nonzero_counts[i], max_seq_len-1)
        nz_idx = torch.nonzero(x_batch[i])[:nz_count].squeeze()
        nz_values = x_batch[i][nz_idx]
        
        values_tensor[i, 0] = 0.0
        values_tensor[i, 1:1+nz_count] = nz_values
        
        genes = ["<cls>"] + [feature_names[idx] for idx in nz_idx.cpu().numpy()]
        genes += ["<pad>"] * (max_seq_len - len(genes))
        
        src_tensor[i] = torch.tensor([
            vocab[gene] if gene in vocab else vocab["<unk>"] for gene in genes
        ], device=device)
        
        padding_mask[i] = torch.tensor([gene == "<pad>" for gene in genes], device=device)
    
    return values_tensor, src_tensor, padding_mask

In [ ]:
def evaluate_single_file(model, file_path, feature_names, vocab):
    """Evaluate individual files and return predicted results"""
    dataset = SingleParquetFileDataset(file_path)
    loader = DataLoader(
        dataset,
        batch_size=256,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )
    
    preds = []
    true_labels = []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Processing {os.path.basename(file_path)}"):
            x = batch['X'].to(device, non_blocking=True)
            cell_types = batch['cell_type']
#             print(cell_types)
            
            # Convert real labels to numbers
            true_labels.extend(cell_types)
#             true_labels.extend([celltype_to_id[cell_type.item()] for cell_type in cell_types])
            
            values, src, mask = preprocess_batch(x, feature_names, vocab, device)
            
            with torch.cuda.amp.autocast(enabled=True):
                outputs = model(src=src, values=values, src_key_padding_mask=mask)
            
            preds.append(outputs["cls_output"].argmax(dim=1).cpu().numpy())
    
    return np.concatenate(preds), np.array(true_labels)  # Return predicted and actual labels

In [ ]:
def load_model(model_path, vocab_path):
    """Load pre-trained model and vocabulary list"""
    vocab = GeneVocab.from_file(vocab_path)
    model = MambaModel(
        ntoken=len(vocab), d_model=512,nlayers=6,
        dropout=0.2, pad_token="<pad>", pad_value=-2,
        input_emb_style="continuous", cell_emb_style="cls"
    )
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    return model, vocab

In [ ]:
if __name__ == "__main__":
    # 1. Load dataset
    df = pd.read_parquet('../../data/merlin_cxg_2023_05_15_sf-log1p/var.parquet')
    feature_names = df['feature_name'].tolist()
    
    # 2. Load model
    model, vocab = load_model(
        "../../model_state/cell_cls_3loss_6layer_sample_final.pth",
        "../../models/vocab.json"
    )
    
    # 3. Retrieve all test files (sorted by number)
    test_dir = "../../data/merlin_cxg_2023_05_15_sf-log1p/test"
    # test_dir = "../../data/merlin_cxg_2023_05_15_sf-log1p/test"
    test_files = sorted(
        [os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith('.parquet')],
        key=lambda x: int(os.path.basename(x).split('.')[1])
    )
    
    # 4. Generate all predicted results and collect real labels
    all_preds = np.array([], dtype=np.int64)
    all_true = np.array([], dtype=np.int64)
    
    for file_path in tqdm(test_files, desc="Processing all files"):
        preds, true = evaluate_single_file(model, file_path, feature_names, vocab)
        all_preds = np.concatenate([all_preds, preds])
        all_true = np.concatenate([all_true, true])
    
    # 5. Save prediction results
    output_path = "./output/mamba_predictions_3loss_sample.pkl"
    with open(output_path, 'wb') as f:
        pickle.dump(all_preds, f)
    
    print(f"\nThe predicted results have been saved to {output_path}")
    print(f"Total number of predicted samples: {len(all_preds)}")
    print(f"Shape of predicted results: {all_preds.shape}")  # Should output (n,)
    
    # 6. Calculate and output accuracy and classification reports
    accuracy = accuracy_score(all_true, all_preds)
    print(f"\nPrediction accuracy: {accuracy:.4f}")
    
    # Generate classification report
    class_report = classification_report(
        all_true, 
        all_preds,
        labels=list(id_to_celltype.keys()),
        target_names=list(id_to_celltype.values()),
        zero_division=0
    )
    
    print("\nClassification report:")
    print(class_report)
    
    # Save classification report to a file
    report_path = output_path.replace('.pkl', '_report.txt')
    with open(report_path, 'w') as f:
        f.write(f"Predicition accuracy: {accuracy:.4f}\n\n")
        f.write("Classification report:\n")
        f.write(class_report)
    
    print(f"\nThe classification report has been saved to {report_path}")